
# Procesamiento de la Carta Marina de Córdoba 2019

## Descripción

Este notebook tiene como objetivo reconstruir y documentar el proceso de extracción de datos de la Carta Marina de Córdoba correspondiente a las elecciones de 2019.

El desarrollo se basa en el trabajo realizado en el repositorio **Carta Marina Córdoba 2017** de avdata99 (https://github.com/avdata99/carta-marina-2017/tree/master).

En esta etapa el trabajo se concentra exclusivamente en la Carta Marina 2019.

## Flujo de trabajo


1. PDF
2. TXT (pdftotext -layout)
3. CSV de mesas/escuelas/electores
4. Geolocalización de escuelas
5. CSV geolocalizado
6. Mapas y análisis


**Productos esperados**

- `LugaresDeVotacion-elecciones-2015.pdf`
- `carta-marina-cordoba-2015.txt`
- `escuelas-elecciones-2015-cordoba.csv`
- `escuelas-elecciones-2015-cordoba-clean.csv` (si requiere correcciones)



0. Dependencias

In [2]:
from google.colab import files
import pandas as pd
from pathlib import Path

In [3]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up libpoppler118:amd64 (22.02.0-2ubuntu0.13) ...
Setting up poppler-

#1. Carga del PDF

In [4]:
archivo = files.upload()

nombre_archivo = next(iter(archivo))

print(f"Archivo cargado: {nombre_archivo}")


Saving 2019.pdf to 2019.pdf
Archivo cargado: 2019.pdf


#2. PDF a TXT

In [5]:
nombre_txt = Path(nombre_archivo).with_suffix(".txt")

!pdftotext -layout "{nombre_archivo}" "{nombre_txt}"

print(f"Archivo generado: {nombre_txt}")

Archivo generado: 2019.txt


#3. TXT a CSV



```
Leer línea
    │
    ├── ¿Es un encabezado o número de página?
    │      └── Sí → ignorar
    │
    ├── ¿Es una sección?
    │      └── Sí → actualizar sección
    │
    ├── ¿Es un circuito?
    │      └── Sí → actualizar circuito e iniciar lectura de establecimientos
    │
    ├── ¿Es el resumen del circuito?
    │      └── Sí → finalizar lectura del circuito
    │
    ├── ¿Está vacía?
    │      └── Sí → ignorar
    │
    ├── ¿Estamos dentro de un circuito?
    │      └── No → continuar
    │
    └── Procesar establecimiento
           ├── Ignorar líneas "asociada a"
           ├── Reconstruir el nombre del establecimiento
           ├── Extraer mesas y electores
           ├── Validar continuidad de mesas
           └── Guardar registro
```





##Inspeccion del TXT

In [6]:
path = f"/content/{nombre_txt}"

with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

lines = raw.splitlines()

print(f"Cantidad total de líneas: {len(lines)}")
print("\nPrimeras 10 líneas:\n")

for i, linea in enumerate(lines[:10], start=1):
    print(i, repr(linea))
print ("...")
print ("...")
print ("...")
print ("...")
for i, linea in enumerate(raw.splitlines()[-10:], start=len(raw.splitlines())-9):
    print(i, repr(linea))

Cantidad total de líneas: 6281

Primeras 10 líneas:

1 '                                                  DISTRITO CORDOBA'
2 '                            ELECCIONES GENERALES - 27 de Octubre de 2019'
3 '                             Informe de Establecimientos, Mesas y Electores Habilitados'
4 ''
5 'Sección 1 - CAPITAL'
6 ''
7 'Circuito:1 - SECCIONAL PRIMERA                                              Cant   Desde / hasta              Cant'
8 '                                                                           Mesas                            Electores'
9 'CENTRO EDUC.NIVEL MEDIO ADULTO - DEAN FUNES 417 CAPITAL - CORDOBA           7       0001 a 0007                 2.415'
10 ''
...
...
...
...
6272 ''
6273 '              Mesas Habilitadas:                                                                     8.750'
6274 ''
6275 '              Electores Habilitados                                                              2.946.060'
6276 ''
6277 ''
6278 ''
6279 ''
6280 'oct 2,

In [7]:
print("Saltos de línea \\n:", raw.count("\n"))
print("Saltos de página \\f:", raw.count("\f"))

print("Con split('\\n'):", len(raw.split("\n")))
print("Con splitlines():", len(raw.splitlines()))

Saltos de línea \n: 6190
Saltos de página \f: 91
Con split('\n'): 6191
Con splitlines(): 6281


##Inicialización de variables

In [33]:
# =====================================
# Inicialización de variables del parser
# =====================================

# Sección electoral (equivale al departamento de la provincia)
seccion_nro = 0
seccion_name = ""

# Circuito electoral dentro de la sección.
# Puede contener letras (ej.: 4A, 4B, 4C), por eso se almacena como texto.
circuito_nro = ""
circuito_name = ""

# Estado actual del parser.
# Cuando vale "escuelas", las líneas leídas corresponden a establecimientos.
imin = ""

# Contador de líneas procesadas del archivo TXT.
cnt = 0

# Contador de errores (heredado del parser original).
errores = 0

# Lista donde se almacenan todos los establecimientos extraídos.
# Cada elemento es un diccionario con la información de un establecimiento.
escuelas = []

# Registra las discontinuidades detectadas en la numeración de mesas.
# Se utiliza como control de calidad, pero no interrumpe la ejecución.
discontinuidades = []

# Última mesa procesada.
# Permite verificar que la numeración de mesas sea continua.
last_mesa = 0

# ============================
# Variables de diagnóstico
# ============================

# Cantidad de números de página ignorados.
paginas_ignoradas = 0

# Cantidad de encabezados "DISTRITO CORDOBA" ignorados.
distritos_ignorados = 0

# Cantidad de encabezados "ELECCIONES 2015" ignorados.
elecciones_ignoradas = 0

# Cantidad de encabezados "Informe de Establecimientos" ignorados.
informes_ignorados = 0

# Almacena las filas que no pudieron procesarse correctamente,
# junto con la información necesaria para su revisión.
filas_con_error = []

print(f"Contador inicial: {cnt}")


Contador inicial: 0


##Ciclo for

In [34]:
for linea in lines:
    cnt += 1

    # Ignorar número de página
    if "Página" in linea or "Pag." in linea:
        paginas_ignoradas += 1
        continue

    # Ignorar encabezados
    if linea.strip() == "DISTRITO CORDOBA":
        distritos_ignorados += 1
        continue

    if "ELECCIONES" in linea:
        elecciones_ignoradas += 1
        continue

    if "Informe de Establecimientos" in linea:
        informes_ignorados += 1
        continue

    if "Mesas" in linea and "Electores" in linea:
        informes_ignorados += 1
        continue

    # Detectar sección
    if linea.startswith("Sección "):
        contenido = linea.removeprefix("Sección ").strip()
        partes = contenido.split("-", 1)

        seccion_nro = int(partes[0].strip())
        seccion_name = partes[1].strip()

        imin = ""
        continue

    # Detectar circuito
    if linea.startswith("Circuito:"):
        contenido = linea.removeprefix("Circuito:").strip()
        partes = contenido.split("-", 1)

        circuito_nro = partes[0].strip()

        # En la misma línea aparecen los títulos de otras columnas
        partes_nombre = partes[1].split("       ")
        circuito_name = partes_nombre[0].strip()

        imin = "escuelas"
        continue

    # Detectar el final del listado de escuelas
    # Las líneas vacías se ignoran, pero no cierran el circuito
    if linea == "":
       continue

    # Solo el resumen cierra realmente el circuito
    if linea.startswith("Resumen del Circuito"):
        imin = ""
        continue

    # Procesar escuelas
    if imin == "escuelas":
        partes = linea.split("    ")

        # Conservar solo fragmentos con contenido
        datos = [x.strip() for x in partes if x.strip() != ""]

        print(f"{cnt} -- {datos}")

        if len(datos) == 1 and "asociada a" in datos[0]:
            continue



        establecimiento = " ".join(datos[:-3])
        cant_mesas = int(datos[-3])

        rango_mesas = datos[-2].split(" a ")
        mesa_desde = int(rango_mesas[0])
        mesa_hasta = int(rango_mesas[1])


        if mesa_desde != last_mesa + 1:
            discontinuidades.append({
              "linea": cnt,
              "esperada": last_mesa + 1,
              "encontrada": mesa_desde,
              "establecimiento": establecimiento
          })
            #raise ValueError(
                #f"Mesa inválida en línea {cnt}: "
                #f"empieza en {mesa_desde}, "
                #f"pero la anterior terminó en {last_mesa}"
            #)

        last_mesa = mesa_hasta

        cant_electores = int(datos[-1].replace(".", ""))

        elem = {
            "seccion_nro": seccion_nro,
            "seccion_name": seccion_name,
            "circuito_nro": circuito_nro,
            "circuito_name": circuito_name,
            "establecimiento": establecimiento.replace(",", "."),
            "cant_mesas": cant_mesas,
            "desde": mesa_desde,
            "hasta": mesa_hasta,
            "electores": cant_electores,
        }

        escuelas.append(elem)
        print(f"Contador inicial: {cnt}")

9 -- ['CENTRO EDUC.NIVEL MEDIO ADULTO - DEAN FUNES 417 CAPITAL - CORDOBA', '7', '0001 a 0007', '2.415']
Contador inicial: 9
11 -- ['ESC NUESTRA SEÑORA DEL HUERTO - BELGRANO 269 CAPITAL - CORDOBA', '12', '0008 a 0019', '4.128']
Contador inicial: 11
13 -- ['COL NAC DE MONSERRAT - OBISPO TREJO 294 CAPITAL - CORDOBA', '18', '0020 a 0037', '6.192']
Contador inicial: 13
15 -- ['ESC SANTA TERESA DE JESUS - OBISPO TREJO Y SANABRIA 160 CAPITAL -', '10', '0038 a 0047', '3.440']
Contador inicial: 15
21 -- ['ESC JUAN BAUTISTA ALBERDI - GRAL PAZ 488 CAPITAL - CORDOBA', '18', '0048 a 0065', '6.176']
Contador inicial: 21
27 -- ['ESC JERONIMO LUIS DE CABRERA - SANTA ROSA 650 CAPITAL - CORDOBA', '15', '0066 a 0080', '5.220']
Contador inicial: 27
29 -- ['IPEM N° 270 GRAL M BELGRANO - DEAN FUNES 850 CAPITAL - CORDOBA', '11', '0081 a 0091', '3.817']
Contador inicial: 29
31 -- ['ESC SANTO TOMAS - CASEROS 745 CAPITAL - CORDOBA', '8', '0092 a 0099', '2.776']
Contador inicial: 31
33 -- ['ESC NORMAL ALEJANDRO 

In [36]:
print(discontinuidades)
print("Establecimientos:", len(escuelas))
print("Discontinuidades:", len(discontinuidades))
print("Última mesa:", last_mesa)
for d in discontinuidades:
    print(d)

[{'linea': 2003, 'esperada': 4116, 'encontrada': 4117, 'establecimiento': 'IPEM N° 253 - ALEM ESQ DOMINGO MATEU CRUZ DEL EJE - CRUZ DEL EJE'}, {'linea': 2428, 'esperada': 4434, 'encontrada': 4436, 'establecimiento': 'ESCUELA J.MARMOL - NAC. UNIDAS 385 VILLA MARIA - VILLA MARIA'}, {'linea': 3732, 'esperada': 6064, 'encontrada': 6065, 'establecimiento': 'ESC NICOLAS AVELLANEDA - VICENTE LOPEZ 538 PUEBLO ALBERDI - RIO CUARTO'}]
Establecimientos: 1224
Discontinuidades: 3
Última mesa: 8750
{'linea': 2003, 'esperada': 4116, 'encontrada': 4117, 'establecimiento': 'IPEM N° 253 - ALEM ESQ DOMINGO MATEU CRUZ DEL EJE - CRUZ DEL EJE'}
{'linea': 2428, 'esperada': 4434, 'encontrada': 4436, 'establecimiento': 'ESCUELA J.MARMOL - NAC. UNIDAS 385 VILLA MARIA - VILLA MARIA'}
{'linea': 3732, 'esperada': 6064, 'encontrada': 6065, 'establecimiento': 'ESC NICOLAS AVELLANEDA - VICENTE LOPEZ 538 PUEBLO ALBERDI - RIO CUARTO'}


In [37]:
print("Electores:", sum(e["electores"] for e in escuelas))

Electores: 2946060


#Dataframe

In [41]:
df_escuelas = pd.DataFrame(escuelas)

display(df_escuelas.head())
display(df_escuelas.tail())

print(df_escuelas.shape)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
0,1,CAPITAL,1,SECCIONAL PRIMERA,CENTRO EDUC.NIVEL MEDIO ADULTO - DEAN FUNES 41...,7,1,7,2415
1,1,CAPITAL,1,SECCIONAL PRIMERA,ESC NUESTRA SEÑORA DEL HUERTO - BELGRANO 269 C...,12,8,19,4128
2,1,CAPITAL,1,SECCIONAL PRIMERA,COL NAC DE MONSERRAT - OBISPO TREJO 294 CAPITA...,18,20,37,6192
3,1,CAPITAL,1,SECCIONAL PRIMERA,ESC SANTA TERESA DE JESUS - OBISPO TREJO Y SAN...,10,38,47,3440
4,1,CAPITAL,2,SECCIONAL SEGUNDA,ESC JUAN BAUTISTA ALBERDI - GRAL PAZ 488 CAPIT...,18,48,65,6176


,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
1219,26,UNION,397A,CORRAL DEL BAJO,ESCUELA VALENTIN ALSINA - CALLE PUBLICA S/N 0 ...,1,8735,8735,5
1220,26,UNION,400,SAN MARCOS SUD,INST JOSE DE SAN MARTIN - ENTRE RIOS 977 SAN M...,5,8736,8740,1605
1221,26,UNION,400,SAN MARCOS SUD,ESC PROV M BUCHARDO - PASAJE AURORA LLANOS DE ...,4,8741,8744,1283
1222,26,UNION,401,SANTA MARIA,ESC PROV PAULA ALBARRACIN - LOS FRESNOS 290 SA...,1,8745,8745,276
1223,26,UNION,402,VIAMONTE,INSTITUTO JUAN B ALBERDI - AVELLANEDA 182 VIAM...,5,8746,8750,1666


(1224, 9)


In [42]:
df_escuelas.sample(20, random_state=42)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
661,11,POCHO,144,CHANCANI,ESC PROV MARIANO MORENO - PUBLICA S/N CHANCANI...,3,5241,5243,1045
220,1,CAPITAL,11K,VILLA GRAL URQUIZA,ESC MUNIC JUAN B JUSTO - INT RAMON B MESTRE S/...,7,2100,2106,2450
155,1,CAPITAL,10I,PARQUE HORIZONTE,PRIMARIA ELPIDIO TORRES - CALLE PUBLICA S/N SM...,12,1504,1515,4200
677,12,PUNILLA,148,CAPILLA DEL MONTE,ESC ESP JUAN MANUEL FERNANDEZ - AV PUEYRREDON ...,10,5312,5321,3460
911,17,ROQUE SAENZ PEÑA,248,LABOULAYE,I.P.E.M. 278-M.ARGENTINAS - AV INDEPENDENCIA 4...,9,6868,6876,3123
168,1,CAPITAL,10L,VILLA EL LIBERTADOR,ESC PTE DR ARTURO U ILLIA - TOTORAL Y ARICA B°...,11,1621,1631,3817
1087,21,SANTA MARIA,322,COSME,ESC PROV OLEGARIO V ANDRADE - PUBLICA S/N COSM...,1,7953,7953,51
113,1,CAPITAL,9C,LOS PARAISOS,ESC PCIA DE MISIONES - FIRPO 2254 B°ZUMARAN - ...,13,1078,1090,4524
605,9,MARCOS JUAREZ,119,ALEJO LEDESMA,INSTITUTO SEC. V.SARSFIELD - HIPILITO YRIGOYEN...,4,4955,4958,1284
536,6,GRAL SAN MARTIN,88,TIO PUJIO,ESC.R.DE ESCALADA - 25 DE MAYO 84 NORTE TIO PU...,10,4424,4433,3312


In [43]:
output_path = "escuelas-elecciones-2019-cordoba.csv"

df_escuelas.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo exportado: {output_path}")

Archivo exportado: escuelas-elecciones-2015-cordoba.csv
